# Downstream analysis of a BayesSpace-enhanced, SpatialWarp-aligned multi-omic object

Unlike the other notebooks in `examples/`, this one doesn't run any alignment itself -- it picks
up **after** it. The object loaded below (`aligned_msi_visium_enhanced.h5ad`) already combines:

1. Visium gene expression for the D1225 section (the same slide used in
   `examples/msi_to_classic_visium.ipynb`),
2. MSI metabolite intensities merged in via `sw.align()` (the 81 metabolites from that same
   notebook, stored here as `msi_<metabolite name>` columns in `.obs`), and
3. **BayesSpace's `enhanceFeatures()`** (an R tool, run outside this package): it subdivides
   each of D1225's 3065 Visium spots into 6 hexagonal sub-spots (18,390 total) and predicts a
   spatially-smoothed value for both gene expression and the MSI metabolite features at that
   finer resolution.

This is the natural next step once alignment is done: alignment gets every modality onto one
coordinate system and one spot index; a spatially-aware smoothing/enhancement tool like
BayesSpace can then be applied across *all* of them at once, RNA and metabolites alike, since
from its point of view they're just "features on a grid."

**Two things worth knowing about this specific file before using it:**

- Enhancement only ran on a subset of genes (computationally, running it genome-wide isn't
  practical). `.var['is.HVG']` flags the 2000 genes with complete, valid enhanced values --
  every other gene is `NaN` in `.X` unless it happens to fall in the larger (~19,209-gene) set
  `.var['enhanceFeatures.rmse']` was computed for. We'll mostly stick to the reliable `is.HVG`
  subset below.
- `.uns['spatial']` here *is* a proper, complete scanpy-native structure (real hires image +
  scalefactors) -- unlike the SpatialData-based objects `spatialwarp`'s own notebooks build
  directly, which only carry a placeholder. So `sc.pl.spatial()` works out of the box on this
  object with no rescaling tricks needed.

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt

## Step 1 -- Load, and reshape the metabolite columns

The MSI metabolite intensities were merged in as `msi_<name>` columns directly in `.obs` (a
flat layout that's convenient for R/Seurat). We pull them into `obsm["metabolite"]` instead --
the same convention every other `spatialwarp` notebook uses for a per-spot feature matrix --
so the analysis code below (and anything copied from `examples/msi_to_classic_visium.ipynb`)
works unchanged.

In [ ]:
adata = ad.read_h5ad("/home/croizer/Documents/02_Analysis/01_MSI_analysis/aligned_msi_visium_enhanced.h5ad")

msi_cols = [c for c in adata.obs.columns if c.startswith("msi_")]
adata.obsm["metabolite"] = adata.obs[msi_cols].rename(columns=lambda c: c[len("msi_") :]).copy()
adata.obs = adata.obs.drop(columns=msi_cols)

print(f"{adata.n_obs} enhanced sub-spots, {adata.obsm['metabolite'].shape[1]} metabolites")
adata

## Step 2 -- Which genes actually have enhanced values?

`.X` is `NaN` for any gene BayesSpace didn't enhance. `is.HVG` marks the reliable, complete
subset (2000 genes) -- use this whenever you need a dense matrix (ranking genes, PCA, ...).
For plotting a single gene you already know by name, any gene with a non-null
`enhanceFeatures.rmse` works too (a larger, ~19,209-gene set).

In [ ]:
hvg_mask = adata.var["is.HVG"] == 1
print(f"{hvg_mask.sum()} of {adata.n_vars} genes have complete, BayesSpace-enhanced values")

## Step 3 -- Spatial plots at enhanced resolution

One gene, one metabolite, side by side, using `sc.pl.spatial` directly -- notice the much finer,
smoother spatial pattern than a plain per-spot plot would show, since every original Visium spot
is now 6 sub-spots. `color=` needs a real column, so the metabolite gets stashed in a throwaway
`.obs` column just for this plot (it isn't part of `.X`/`var_names`).

In [ ]:
gene = "APOH"
metabolite = "Linoleic acid"

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sc.pl.spatial(adata, color=gene, ax=axes[0], show=False, cmap="viridis", title=f"{gene} (enhanced)")

vmax = np.nanpercentile(adata.obsm["metabolite"][metabolite], 99)
adata.obs["_metabolite_tmp"] = adata.obsm["metabolite"][metabolite].values
sc.pl.spatial(adata, color="_metabolite_tmp", ax=axes[1], show=False, cmap="PiYG", vmax=vmax, title=f"{metabolite} (enhanced)")
del adata.obs["_metabolite_tmp"]

plt.tight_layout()
plt.show()

## Step 4 -- Differential metabolites per cluster

BayesSpace already computed its own spatially-aware clustering at this enhanced resolution
(`obs["spatial.cluster"]`) -- we reuse it directly rather than re-running our own PCA/Leiden,
since it's spatially informed in a way plain expression-based clustering isn't. Same two-way
clustered heatmap recipe as `examples/msi_to_classic_visium.ipynb`, over *all* metabolites.

In [ ]:
adata.obs["spatial.cluster"] = adata.obs["spatial.cluster"].astype(str).astype("category")

metabo_df = adata.obsm["metabolite"].copy()
metabo_df["cluster"] = adata.obs["spatial.cluster"].values
cluster_means = metabo_df.groupby("cluster").mean()

sns.clustermap(cluster_means, cmap="PiYG", standard_scale=1, figsize=(14, 6), dendrogram_ratio=(0.05, 0.05))
plt.show()

## Step 5 -- Differential genes per cluster

Restricted to the `is.HVG` subset (Step 2) so every gene has a real, complete value -- `NaN`
genes would otherwise break `rank_genes_groups`/the per-cluster mean silently.

One gotcha: this file has `adata.raw` set to the full, un-enhanced 38,479-gene matrix (a
Seurat/scanpy convention for keeping pre-processing data around), and `sc.tl.rank_genes_groups`
defaults to `use_raw=True` whenever `.raw` exists -- silently ranking against the wrong (raw,
unenhanced) matrix instead of `adata_hvg`'s enhanced one, and returning gene names that aren't
even in the `is.HVG` subset. `use_raw=False` forces it to use `adata_hvg.X` as intended.

In [ ]:
adata_hvg = adata[:, hvg_mask].copy()
adata_hvg.obs["cluster"] = adata.obs["spatial.cluster"].values

sc.tl.rank_genes_groups(adata_hvg, groupby="cluster", method="wilcoxon", use_raw=False)

top_n = 5
top_genes = []
for cluster in adata_hvg.obs["cluster"].cat.categories:
    top_genes += list(adata_hvg.uns["rank_genes_groups"]["names"][cluster][:top_n])
top_genes = list(dict.fromkeys(top_genes))  # dedupe, keep rank order

gene_df = pd.DataFrame(adata_hvg[:, top_genes].X, columns=top_genes, index=adata_hvg.obs_names)
gene_df["cluster"] = adata_hvg.obs["cluster"].values
cluster_means_genes = gene_df.groupby("cluster").mean()

sns.clustermap(cluster_means_genes, cmap="coolwarm", standard_scale=1, figsize=(14, 6), dendrogram_ratio=(0.05, 0.05))
plt.show()

## Step 6 -- Clusters in space

A final look at where BayesSpace's clusters actually sit on the tissue, at full enhanced
resolution.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
sc.pl.spatial(adata, color="spatial.cluster", ax=ax, show=False, title="BayesSpace spatial clusters (enhanced resolution)")
plt.show()